# Longitudinal invariant $J$ — split integration past the bounce singularity

We estimate

$$
J \propto \int_0^{s^*} \sqrt{B^* - B(s)}\, ds
$$

with $s=0$ on the magnetic ridge and $s^*$ the bounce point where $B(s^*) = B^*$. Near $s^*$ the integrand is singular; we split at $s_{\mathrm{mid}} = s^*/2$ and on $[s_{\mathrm{mid}}, s^*]$ use $t = \sqrt{B^* - B(s)}$, so $2t\,dt = -B'(s)\,ds$ and

$$
\int_{s_{\mathrm{mid}}}^{s^*} \sqrt{B^* - B(s)}\, ds
 = \int_0^{t_{\mathrm{mid}}} \frac{2 t^2}{|B'(s(t))|}\, dt,
\quad t_{\mathrm{mid}} = \sqrt{B^* - B(s_{\mathrm{mid}})}.
$$

The notebook loads coils, traces $\hat{b}$, locates $s^*$, then integrates both halves with `QuadGK`.

### Cell 1 — Setup and initialization

Packages: ODE solver stack (`DifferentialEquations`), automatic differentiation (`ForwardDiff`), adaptive quadrature (`QuadGK`), spline interpolation (`DataInterpolations`), and `CairoMakie` for figures. `Coil.jl` supplies `eval_B` and coil IO.

**Environment:** add any missing packages once, e.g. `using Pkg; Pkg.add(["DifferentialEquations", "DataInterpolations", "QuadGK"])`.

In [9]:
using Pkg
Pkg.activate("c:/Users/danie/Stellarator_Isodrasticity")
using DifferentialEquations
using LinearAlgebra
using ForwardDiff
using StaticArrays
using QuadGK
using GLMakie
using DataInterpolations

const REPO = raw"C:/Users/danie/Stellarator_Isodrasticity"
cd(REPO)

include(joinpath(REPO, "Coil.jl"))

const COIL_FILE = joinpath(REPO, "landreman_paul.json")
const NQUAD = 64

coils = get_coils_from_file(Float64, COIL_FILE, NQUAD)
length(coils)

  Activating project at `c:\Users\danie\Stellarator_Isodrasticity`


16

### Cell 2 — Core physics: $|B|$, its gradient, and Dual-safe Biot–Savart

`eval_B` comes from `Coil.jl`. For `ForwardDiff.gradient` we evaluate $|B|$ through a type-agnostic Biot–Savart loop (same pattern as `magnetic_ridge_3.jl`) so `Dual` numbers propagate without signature errors.

In [10]:
"""
    B_agnostic(cs, x)

Vector `B` at `x` for any scalar type (Float64 or ForwardDiff.Dual).
"""
function B_agnostic(cs::Vector{Coil{T}}, x::AbstractVector) where {T}
    Bacc = zeros(eltype(x), 3)
    for c in cs
        for (ri, Idri) in zip(c.rs, c.Idrs)
            δ = SVector(ri[1] - x[1], ri[2] - x[2], ri[3] - x[3])
            d = sqrt(δ[1]^2 + δ[2]^2 + δ[3]^2)
            Bacc .+= cross(δ, Idri) ./ (d^3)
        end
    end
    return SVector{3,eltype(x)}(Bacc[1], Bacc[2], Bacc[3])
end

"""
    norm_B(coils, r)

Scalar |B(r)|.
"""
norm_B(coils::Vector{Coil{T}}, r::AbstractVector) where {T} = norm(B_agnostic(coils, r))

"""
    grad_B(coils, r)

Gradient of |B| with respect to Cartesian r.
"""
function grad_B(coils::Vector{Coil{T}}, r::AbstractVector) where {T}
    x = SVector{3,Float64}(r[1], r[2], r[3])
    ForwardDiff.gradient(z -> norm_B(coils, z), x)
end

# Quick sanity check
r0 = SA[1.2, 0.05, 0.0]
norm_B(coils, r0), grad_B(coils, r0)

(1.0232102886765506, [-1.5431946226828595, -0.08053888504864837, -0.050334109710863426])

### Cell 2.5 — Surface parameterization bridge

Load the polynomial–Fourier ridge fit (`coeffs`, `M`, `N`) from Set 2 and define helpers that map cylindrical `(R, \phi)` to ridge-consistent Cartesian starts `SA[x, y, z]`.

### Cell 3 — Field-line tracing: $d\mathbf{r}/ds = \hat{b}$

Independent variable $s$ is arc length. State $\mathbf{u} = (x,y,z)$. We integrate with `Tsit5` and dense output on a uniform grid in $s$ so $B(s)$ is well resolved for splines.

In [11]:
using JLD2

const RIDGE_COEFFS_FILE = "C:/Users/danie/Stellarator_Isodrasticity/data/ridge_coeffs.jld2"

coeffs, M, N = jldopen(RIDGE_COEFFS_FILE, "r") do f
    Vector{Float64}(f["coeffs"]), Int(f["M"]), Int(f["N"])
end

"""
    evaluate_ridge_z(R, phi, coeffs, M, N)

Evaluate z(R, phi) using the polynomial–Fourier basis
z = sum_{m=0}^M R^m [A_{m,0} + sum_{n=1}^N (A_{m,n} cos(n phi) + B_{m,n} sin(n phi))].
"""
function evaluate_ridge_z(R::Float64, phi::Float64, coeffs::AbstractVector{<:Real}, M::Int, N::Int)
    z = 0.0
    k = 1

    # n = 0 block: A_{m,0}
    for m in 0:M
        z += coeffs[k] * (R^m)
        k += 1
    end

    # n >= 1 blocks: A_{m,n} cos(n phi), then B_{m,n} sin(n phi)
    for n in 1:N
        cn = cos(n * phi)
        sn = sin(n * phi)
        for m in 0:M
            Rm = R^m
            z += coeffs[k] * Rm * cn
            k += 1
            z += coeffs[k] * Rm * sn
            k += 1
        end
    end

    return z
end

"""
    get_starting_point(R, phi)

Return ridge-consistent initial point SA[x, y, z] from cylindrical (R, phi).
"""
function get_starting_point(R::Float64, phi::Float64)
    z = evaluate_ridge_z(R, phi, coeffs, M, N)
    x = R * cos(phi)
    y = R * sin(phi)
    SA[x, y, z]
end

# Quick bridge sanity check
get_starting_point(1.15, 0.3)

3-element SVector{3, Float64} with indices SOneTo(3):
 1.0986369624944468
 0.33984823766054045
 0.05909155238171293

### Cell 3 — Phase 1 Visual Proof (Isodrasticity Contours)

This cell reconstructs a trimmed analytical ridge surface using a nearest-point mask from raw `(new_rx, new_ry)` samples and overlays magnetic-field-strength contours to visually check whether `|B|` level sets align with the segmented ridge geometry.

In [22]:
ridge_mask_file = joinpath(REPO, "data", "magnetic_ridge3", "magnetic_ridge_3.jld2")
new_rx, new_ry = jldopen(ridge_mask_file, "r") do f
    rx_key = haskey(f, "new_rx") ? "new_rx" : "ridge_x"
    ry_key = haskey(f, "new_ry") ? "new_ry" : "ridge_y"
    Vector{Float64}(f[rx_key]), Vector{Float64}(f[ry_key])
end

R_mask = hypot.(new_rx, new_ry)
R_min, R_max = extrema(R_mask)

nR = 320
nphi = 480
R_vals = range(R_min, R_max; length = nR)
phi_vals = range(-π, π; length = nphi)

R_grid = [R for R in R_vals, _ in phi_vals]
Phi_grid = [phi for _ in R_vals, phi in phi_vals]
X_surface = R_grid .* cos.(Phi_grid)
Y_surface = R_grid .* sin.(Phi_grid)

function build_masked_surfaces!(Z_surface, B_mag_surface, X_surface, Y_surface, R_grid, Phi_grid, new_rx, new_ry, th)
    fill!(Z_surface, NaN)
    fill!(B_mag_surface, NaN)
    th2 = th^2

    nR, nphi = size(Z_surface)
    for i in 1:nR
        for j in 1:nphi
            x = X_surface[i, j]
            y = Y_surface[i, j]
            d2_min = minimum((new_rx .- x).^2 .+ (new_ry .- y).^2)

            if d2_min < th2
                R = R_grid[i, j]
                phi = Phi_grid[i, j]
                z = evaluate_ridge_z(R, phi, coeffs, M, N)
                Z_surface[i, j] = z
                B_mag_surface[i, j] = norm_B(coils, SA[x, y, z])
            end
        end
    end

    count(.!isnan.(Z_surface))
end

Z_surface = fill(NaN, nR, nphi)
B_mag_surface = fill(NaN, nR, nphi)

# Looser mask for less fragmented trimmed geometry
threshold_candidates = (0.10, 0.14, 0.20, 0.28)
used_threshold = threshold_candidates[1]
valid_count = 0
for th in threshold_candidates
    valid_count = build_masked_surfaces!(Z_surface, B_mag_surface, X_surface, Y_surface, R_grid, Phi_grid, new_rx, new_ry, th)
    used_threshold = th
    valid_count > 0 && break
end

valid_count > 0 || error("Phase 1 mask produced zero valid points. Increase threshold or check mask dataset keys.")
valid_B = B_mag_surface[.!isnan.(B_mag_surface)]
b_min, b_max = extrema(valid_B)
levels_B = length(valid_B) > 15 ? range(b_min, b_max; length = 15) : 8
banded_cmap = cgrad(:jet, 15, categorical = true)

fig_phase1 = Figure(size = (1100, 760))
ax_phase1 = Axis3(
    fig_phase1[1, 1];
    title = "Phase 1 Isodrasticity Check: Σ⁻ painted by |B|",
    xlabel = "x",
    ylabel = "y",
    zlabel = "z",
    aspect = :data,
)

# Paint |B| directly on the physical surface geometry with tight color scaling
surface!(
    ax_phase1,
    X_surface,
    Y_surface,
    Z_surface;
    color = B_mag_surface,
    colormap = banded_cmap,
    colorrange = (b_min, b_max),
    nan_color = RGBAf(0, 0, 0, 0),
    shading = true,
)

# Optional floor-projected contours to inspect contour closure
z_floor = minimum(Z_surface[.!isnan.(Z_surface)]) - 0.08
contour!(
    ax_phase1,
    X_surface,
    Y_surface,
    B_mag_surface;
    levels = levels_B,
    colormap = banded_cmap,
    linewidth = 2,
    transformation = (:xy, z_floor),
)

println("Mask threshold used: ", used_threshold, " | valid grid points: ", valid_count, " / ", length(Z_surface))
println("|B| range on valid surface: [", round(b_min; digits = 6), ", ", round(b_max; digits = 6), "]")
fig_phase1

Mask threshold used: 0.1 | valid grid points: 112591 / 153600
|B| range on valid surface: [0.785988, 6.011367]


In [18]:
function field_rhs!(du, u, p, s)
    B = B_agnostic(p.coils, u)
    nb = norm(B)
    nb < eps(Float64) * 1e8 && (du .= 0; return)
    b = B / nb
    du[1], du[2], du[3] = b[1], b[2], b[3]
    nothing
end

"""
    trace_field_line(coils, R, phi, s_max; n = 4000)

Build a ridge-consistent start point from `(R, phi)`, then return `(s, B_mag)`
with `s` uniform on `[0, s_max]` and `B_mag[i] = |B(u(s_i))|`.
"""
function trace_field_line(coils, R::Float64, phi::Float64, s_max::Float64; n::Int = 4000)
    u0 = get_starting_point(R, phi)
    u0v = collect(Float64, u0)
    prob = ODEProblem(field_rhs!, u0v, (0.0, s_max), (; coils))
    st = range(0.0, s_max; length = n)
    sol = solve(
        prob,
        Tsit5();
        saveat = st,
        abstol = 1e-10,
        reltol = 1e-10,
        save_everystep = false,
    )
    s = collect(sol.t)
    B_mag = Float64[]
    sizehint!(B_mag, length(sol.u))
    for u in sol.u
        push!(B_mag, norm_B(coils, u))
    end
    return s, B_mag
end

# Example trace from magnetic ridge parameterization
phi = 0.3
R0 = 1.15
s_max = 80.0
u_start = get_starting_point(R0, phi)
s_data, B_data = trace_field_line(coils, R0, phi, s_max; n = 3000)

fig_trace = Figure(size = (900, 360))
ax_trace = Axis(
    fig_trace[1, 1];
    xlabel = "s (arc length)",
    ylabel = "|B|",
    title = "Field-line trace diagnostic: |B(s)|",
)
lines!(ax_trace, s_data, B_data; color = :black, linewidth = 1.6)
fig_trace

(u_start, length(s_data), extrema(B_data))

([1.0986369624944468, 0.33984823766054045, 0.05909155238171293], 3000, (0.9579207248593996, 1.0558787899103472))

### Cell 4 — Bounce point $s^*$ where $B(s^*) = B^*$

On the discrete trace, build a smooth spline $B(s)$, then bracket and refine the root of $B(s) - B^* = 0$. We assume there is a solution with $s^* > 0$ and that $B^*$ lies between $B(0)$ and $\max B$ along the segment of interest (adjust `B_star` otherwise).

In [13]:
"""
    spline_B_of_s(s_data, B_data)

Cubic spline with nodes sorted by `s`.
"""
function spline_B_of_s(s_data::Vector{Float64}, B_data::Vector{Float64})
    p = sortperm(s_data)
    s_sorted = s_data[p]
    B_sorted = B_data[p]
    # strictly increasing knots for CubicSpline (merge near-duplicates)
    s_out = Float64[s_sorted[1]]
    B_out = Float64[B_sorted[1]]
    for i in 2:length(s_sorted)
        if s_sorted[i] > s_out[end] + 1e-12
            push!(s_out, s_sorted[i])
            push!(B_out, B_sorted[i])
        else
            B_out[end] = B_sorted[i]
        end
    end
    length(s_out) ≥ 4 || error("Need at least 4 spline knots after deduplication")
    CubicSpline(B_out, s_out)
end

B_of_s(S::CubicSpline, s::Float64) = S(s)

"""
    B_prime_from_spline(S, s)

Derivative dB/ds using `DataInterpolations.derivative`.
"""
B_prime_from_spline(S::CubicSpline, s::Float64) = DataInterpolations.derivative(S, s)

"""
    find_bounce_arclength(s_data, B_data, B_star)

Returns `s_star` with `B(s_star) ≈ B_star` (first crossing found by scanning).
"""
function find_bounce_arclength(s_data::Vector{Float64}, B_data::Vector{Float64}, B_star::Float64)
    S = spline_B_of_s(s_data, B_data)
    smin, smax = extrema(s_data)
    g(s) = B_of_s(S, s) - B_star
    # bracket: look for sign change on spline grid
    ss = range(smin, smax; length = max(200, length(s_data)))
    local a::Float64, b::Float64
    a, b = smin, smax
    found = false
    gs = g.(ss)
    for i in 1:(length(ss)-1)
        if gs[i] == 0
            return ss[i]
        end
        if gs[i] * gs[i+1] < 0
            a, b = ss[i], ss[i+1]
            found = true
            break
        end
    end
    found || error("Could not bracket B(s) = B_star; check B_star lies between min/max B along trace.")
    for _ in 1:60
        m = (a + b) / 2
        if g(a) * g(m) ≤ 0
            b = m
        else
            a = m
        end
        if abs(b - a) < 1e-12 * (1 + abs(m))
            return (a + b) / 2
        end
    end
    return (a + b) / 2
end

# Pick a bounce level above the starting |B|
B_star = B_data[1] + 0.35 * (maximum(B_data) - B_data[1])
S_bounce = spline_B_of_s(s_data, B_data)
s_star = find_bounce_arclength(s_data, B_data, B_star)

fig_bounce = Figure(size = (900, 360))
ax_bounce = Axis(
    fig_bounce[1, 1];
    xlabel = "s (arc length)",
    ylabel = "|B|",
    title = "Bounce root diagnostic: B(s) with B* and s*",
)
lines!(ax_bounce, s_data, B_data; color = :black, linewidth = 1.4)
hlines!(ax_bounce, [B_star]; color = :blue, linestyle = :dash)
vlines!(ax_bounce, [s_star]; color = :red, linestyle = :dash)
scatter!(ax_bounce, [s_star], [B_of_s(S_bounce, s_star)]; color = :red, markersize = 10)
fig_bounce

(B_star, s_star, B_of_s(S_bounce, s_star))

(1.0510851103400516, 13.675507217945011, 1.0510851103400092)

### Cell 5 — Split integration for $J \propto \int_0^{s^*} \sqrt{B^* - B(s)}\, ds$

- **Left:** $\int_0^{s_{\mathrm{mid}}} \sqrt{B^* - B(s)}\, ds$ with $s_{\mathrm{mid}} = s^*/2$.
- **Right:** $\int_{t_{\mathrm{mid}}}^{0} \frac{2 t^2}{|B'(s)|}\, dt$ with $t = \sqrt{B^* - B(s)}$, mapping each $t$ to $s \in [s_{\mathrm{mid}}, s^*]$ by solving $B(s) = B^* - t^2$ (bisection, monotone segment).

We use the same cubic spline for $B(s)$ and its analytic derivative from `DataInterpolations`.

In [14]:
"""
    s_from_t_target(S, B_star, t, slo, shi)

Find `s ∈ [slo, shi]` such that `B(s) = B_star - t^2`.
"""
function s_from_t_target(S::CubicSpline, B_star::Float64, t::Float64, slo::Float64, shi::Float64)
    target = B_star - t^2
    h(s) = B_of_s(S, s) - target
    ha, hb = h(slo), h(shi)
    ha * hb ≤ 0 || error("B(s)=B_star-t^2 not bracketed; check monotonicity on [slo, shi]")
    a, b = slo, shi
    for _ in 1:80
        m = (a + b) / 2
        hm = h(m)
        abs(hm) < 1e-14 * (1 + abs(target)) && return m
        if ha * hm ≤ 0
            b, hb = m, hm
        else
            a, ha = m, hm
        end
        abs(b - a) < 1e-13 * (1 + abs(m)) && return (a + b) / 2
    end
    return (a + b) / 2
end

"""
    calculate_J(s_data, B_data, B_star; atol=1e-8)

Returns `(J_estimate, s_star, s_mid, left_part, right_part, S)`.
"""
function calculate_J(s_data::Vector{Float64}, B_data::Vector{Float64}, B_star::Float64; atol::Float64 = 1e-8)
    S = spline_B_of_s(s_data, B_data)
    s_star = find_bounce_arclength(s_data, B_data, B_star)
    s_mid = s_star / 2

    B0 = B_of_s(S, 0.0)
    B_star - B0 < 0 && error("Need B_star ≥ B(0) for nonnegative integrand on [0,s*]")

    smin, smax = extrema(s_data)
    (0 ≤ s_mid ≤ s_star) || error("s_mid out of range")

    left_integrand(s) = sqrt(max(B_star - B_of_s(S, s), 0.0))
    L, el = quadgk(left_integrand, 0.0, s_mid; atol = atol, rtol = atol)

    t_mid = sqrt(max(B_star - B_of_s(S, s_mid), 0.0))
    function right_integrand(t)
        t ≤ 0 && return 0.0
        sc = s_from_t_target(S, B_star, t, s_mid, s_star)
        Bp = B_prime_from_spline(S, sc)
        den = max(abs(Bp), 1e-30)
        return 2 * t^2 / den
    end
    # integrate t from t_mid down to 0 ⇒ swap limits
    R, er = quadgk(right_integrand, 0.0, t_mid; atol = atol, rtol = atol)

    J = L + R
    return (; J, s_star, s_mid, left = L, right = R, spline = S, t_mid, el, er)
end

res = calculate_J(s_data, B_data, B_star)
(res.J, res.left, res.right, res.s_star, res.s_mid)

(2.975535817103541, 1.540757810746555, 1.4347780063569864, 13.675507217945011, 6.8377536089725055)

### Cell 6 — Visualization

**Plot 1:** $B(s)$ with vertical lines at $0$, $s_{\mathrm{mid}}$, $s^*$.

**Plot 2:** Phase-space style curves $v_\parallel = \pm \sqrt{B^* - B(s)}$ vs $s$ (only real where $B^* \geq B(s)$).

In [16]:
Splot = spline_B_of_s(s_data, B_data)
ss_fine = range(s_data[1], s_data[end]; length = 1200)
Bfine = [B_of_s(Splot, s) for s in ss_fine]

fig1 = Figure(size = (900, 420))
ax1 = Axis(
    fig1[1, 1];
    xlabel = "s (arc length)",
    ylabel = "|B|",
    title = "|B(s)| along field line — B* = $(round(B_star; digits=4))",
)
lines!(ax1, collect(ss_fine), Bfine; color = :black, linewidth = 1.5)
vlines!(ax1, [0.0]; color = :gray, linestyle = :dash, linewidth = 2)
vlines!(ax1, [res.s_mid]; color = :orange, linestyle = :dash, linewidth = 2)
vlines!(ax1, [res.s_star]; color = :red, linestyle = :dash, linewidth = 2)
fig1

# Phase-space style: v_parallel = ± sqrt(B* - B(s))
v_plus = Float64[]
v_minus = Float64[]
for s in ss_fine
    Bs = B_of_s(Splot, s)
    dv = B_star - Bs
    if dv ≥ 0
        v = sqrt(dv)
        push!(v_plus, v)
        push!(v_minus, -v)
    else
        push!(v_plus, NaN)
        push!(v_minus, NaN)
    end
end

fig2 = Figure(size = (900, 420))
ax2 = Axis(fig2[1, 1]; xlabel = "s", ylabel = "±√(B*−B(s))", title = "Phase-space style (parallel velocity proxy)")
lp = lines!(ax2, collect(ss_fine), v_plus; color = :blue, linewidth = 1.5)
lm = lines!(ax2, collect(ss_fine), v_minus; color = :teal, linewidth = 1.5)
vlines!(ax2, [0.0]; color = :gray, linestyle = :dash)
vlines!(ax2, [res.s_mid]; color = :orange, linestyle = :dash)
vlines!(ax2, [res.s_star]; color = :red, linestyle = :dash)
axislegend(ax2, [lp, lm], ["+", "−"]; position = :rt)
fig2